In [1]:
MAIN_PROMPT = """Your purpose is to extract faithful information from genetic research papers.

Upon being presented with a research paper, you are to produce a output in the following format, no additional summary or explanation is needed:
Symbol: (An abbreviation for the gene, composed of capital letters and numbers)
Name of gene: (The full name of the gene)
Disease: stated in database
CHD Phenotype: stated in database List 1 phenotype per line, starting with “-” and separated by “;”

Here is an good example output:
Symbol: ACVR2B
Name of gene: activin A receptor type 2B
Disease: Heterotaxy, visceral, 4
CHD Phenotype:
-Atrial septal defect;
-Ventricular septal defect;
-Atrioventricular septal defect;
-Pulmonary atresia;
-Pulmonic stenosis;
-Mitral atresia;
-Transposition of the great arteries;
-Double outlet right ventricle heterotaxy;
-Total anomalous pulmonary venous return;
"""

In [7]:
# Step 1:
import os
import json
import requests

# pdf file into LLM's response 
def synthesize_evidence(pdf_path, gene_name):
    query = "\nThe gene name to search for is %s and this is its research paper text:\n" % gene_name
    # 1️⃣ 读取 PDF 文件并转成 base6
    with open(pdf_path, "rb") as f:
        #pdf_data = base64.standard_b64encode(f.read()).decode("utf-8")
        pdf_data = process_pdf(pdf_path)
        
        url = "https://api.chatanywhere.tech/v1/chat/completions"
            
        # 设置请求头
        headers = {
                "Content-Type": "application/json",
                "Authorization": "sk-AEWLjMo2GHhyVBqrI5Zk85kNtMOHltbGd100nKXvZclh4fZE"  # 替换为你的API密钥  sk-OMaDNShpHHWKHxoIk3AAXfFrLKSgPv5UulbOtxT3O9amfLZu
            }    
        data = {
            "model": "gemini-2.5-pro",  # 模型名称
            "messages": [
                {"role": "user", "content": MAIN_PROMPT + query + pdf_data}
            ],
        }
            
        # 发送POST请求
        response = requests.post(url, headers=headers, data=json.dumps(data))
            
        # 解析响应
        if response.status_code == 200:
            result = response.json()
            generated_text = result['choices'][0]['message']['content']
            print("生成的文本：", generated_text)
            return generated_text
            
        else:
            print("请求失败，状态码：", response.status_code)
            print("错误信息：", response.text) 
            return None

In [3]:
# Step 2:
# response object into json format
import re
def parse_response(output_text, MODEL):
    # Extract using regex
    #print("parsing response...extracting...")
    #print(output_text)
    symbol = re.search(r'Symbol:\s*([^\n\r]*)', output_text)
    gene_name = re.search(r'Name of gene:\s*([^\n\r]*)', output_text)
    disease = re.search(r'Disease:\s*([^\n\r]*)', output_text)
    
    # Extract CHD Phenotypes
    #phenotypes = re.findall(r'-\s*(.+?);', output_text)
    #chd_phenotype = '; '.join([p.strip() for p in phenotypes])
    
    chd_match = re.search(r'CHD Phenotype:\s*(.*?)(?=\n\w+:|$)', output_text, re.DOTALL)
    if chd_match:
        chd_content = chd_match.group(1)
        # 提取所有以-开头的行，不管后面有没有分号
        chd_phenotypes = re.findall(r'-\s*([^\n]*)', chd_content)
        chd_phenotypes =  "; ".join(p.split(';')[0].strip() for p in chd_phenotypes if p.strip())
        #print("Method 3a:", phenotypes)
    extracted_symbol_column = "Symbol (%s)" % MODEL
    extracted_genename_column = "Gene Name (%s)" % MODEL
    extracted_disease_column = "Disease (%s)" % MODEL
    extracted_phenotype_column = "CHD Phenotype (%s)" % MODEL
    
    # Create data dictionary
    data = {
        extracted_symbol_column: symbol.group(1).strip() if symbol else '',
        extracted_genename_column: gene_name.group(1).strip() if gene_name else '',
        extracted_disease_column: disease.group(1).strip() if disease else '',
        extracted_phenotype_column: chd_phenotypes
    }
    # print extracted columns in json output {column: value} format
    print(data)
    return data

In [4]:
import fitz
import re

def process_pdf(pdf_path):
    """增强版：包含更多Reference变体"""
    print(f"📖 正在处理: {pdf_path}")
    
    try:
        doc = fitz.open(pdf_path)
        full_text = ""
        
        # 包含更多可能的Reference变体
        reference_patterns = [
            r'References?\r',
            r'References?\n',
            r'References?\s',  # 空格
            r'REFERENCES?\r', 
            r'REFERENCES?\n',
            r'REFERENCES?\s',
            r'Reference?\r',
            r'Reference?\n',
            r'Reference?\s',
            r'REFERENCE?\r',
            r'REFERENCE?\n',
            r'REFERENCE?\s',
        ]
        
        pattern = '|'.join(reference_patterns)
        
        for page_num in range(len(doc)):
            page = doc[page_num]
            page_text = page.get_text()
            
            match = re.search(pattern, page_text, re.IGNORECASE)
            
            if match:
                print(f"🔍 在第 {page_num+1} 页检测到参考文献标题: {match.group()}")
                text_before = page_text[:match.start()].strip()
                if text_before:
                    full_text += text_before + "\n"
                break
            else:
                full_text += page_text + "\n"
        
        doc.close()
        return full_text.strip()
        
    except Exception as e:
        print(f"❌ 处理PDF失败: {e}")
        return None

In [5]:
import pandas as pd
import time
import json
import os
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception, before_sleep_log
import logging
import re
import base64
import math
# 设置日志
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def extract_wait_time_from_error(error_msg):
    """从错误信息中精确提取等待时间"""
    wait_match = re.search(r'try again in ([0-9.]+)s', error_msg)
    if wait_match:
        return float(wait_match.group(1))
    return None

# 智能重试装饰器 - 根据您的限制优化
@retry(
    stop=stop_after_attempt(6),  # 最多重试6次（比之前少，因为等待时间短）
    wait=wait_exponential(multiplier=2, min=32, max=60),  # 更激进的退避：4, 8, 16, 32, 64秒
    retry=retry_if_exception(lambda e: "rate_limit_exceeded" in str(e) or "Rate limit" in str(e)),
    before_sleep=lambda retry_state: logger.info(f"⏳ 速率限制，第 {retry_state.attempt_number} 次重试，等待 {retry_state.next_action.sleep} 秒...")
)
def synthesize_evidence_robust(pdf_path, gene_name):
    """带智能重试的API调用"""
    try:
        return synthesize_evidence(pdf_path, gene_name)
    except Exception as e:
        error_msg = str(e)
        wait_time = extract_wait_time_from_error(error_msg)
        
        if wait_time and wait_time < 10:  # 如果等待时间很短
            print(f"🔄 短时间等待 {wait_time + 32:.1f} 秒后重试...")
            time.sleep(wait_time + 1)  # 多加1秒缓冲
            # 直接重新抛出异常，让tenacity继续处理
            raise e
        else:
            # 让tenacity的指数退避处理
            raise e

class RateLimitManager:
    """速率限制管理器"""
    
    def __init__(self, tokens_per_minute=30000, safety_margin=0.8):
        self.tokens_per_minute = tokens_per_minute
        self.safety_margin = safety_margin  # 安全边际
        self.request_times = []
        self.estimated_tokens_per_request = 5000  # 根据您的报错估算
        
        # 计算安全限制
        self.safe_tokens_per_minute = tokens_per_minute * safety_margin
        self.max_requests_per_minute = int(self.safe_tokens_per_minute / self.estimated_tokens_per_request)
        
        print(f"🔒 速率限制配置:")
        print(f"   - 限额: {tokens_per_minute} tokens/分钟")
        print(f"   - 安全限额: {self.safe_tokens_per_minute:.0f} tokens/分钟")
        print(f"   - 估算每个请求: {self.estimated_tokens_per_request} tokens")
        print(f"   - 最大安全请求: {self.max_requests_per_minute} 个/分钟")
    
    def should_wait(self):
        """检查是否需要等待"""
        current_time = time.time()
        # 移除1分钟前的记录
        self.request_times = [t for t in self.request_times if current_time - t < 60]
        
        if len(self.request_times) >= self.max_requests_per_minute:
            oldest_request = self.request_times[0]
            wait_time = 60 - (current_time - oldest_request)
            return max(wait_time, 32)  # 最少等待5秒
        
        return 0
    
    def record_request(self):
        """记录请求时间"""
        self.request_times.append(time.time())
    
    def get_current_rate(self):
        """获取当前请求速率"""
        current_time = time.time()
        recent_requests = [t for t in self.request_times if current_time - t < 60]
        return len(recent_requests)

def load_progress(progress_file):
    """加载进度文件"""
    if os.path.exists(progress_file):
        with open(progress_file, 'r', encoding='utf-8') as f:
            return json.load(f)
    return {'processed_files': [], 'all_concat_rows': []}

def save_progress(progress_file, processed_files, all_concat_rows):
    """保存进度"""
    progress = {
        'processed_files': processed_files,
        'all_concat_rows': all_concat_rows,
        'timestamp': time.time()
    }
    with open(progress_file, 'w', encoding='utf-8') as f:
        json.dump(progress, f, ensure_ascii=False, indent=2)

def append_to_excel(existing_file, new_data_df, all_columns):
    """追加数据到Excel文件"""
    if os.path.exists(existing_file):
        try:
            existing_df = pd.read_excel(existing_file, engine='openpyxl')
            combined_df = pd.concat([existing_df, new_data_df], ignore_index=True)
            combined_df = combined_df.reindex(columns=all_columns, fill_value='')
            combined_df.to_excel(existing_file, index=False, engine='openpyxl')
            print(f"✅ 已追加 {len(new_data_df)} 行到现有Excel文件")
        except Exception as e:
            print(f"❌ 读取现有Excel文件失败，创建新文件: {e}")
            new_data_df.to_excel(existing_file, index=False, engine='openpyxl')
    else:
        new_data_df.to_excel(existing_file, index=False, engine='openpyxl')
        print(f"✅ 创建新Excel文件，写入 {len(new_data_df)} 行")

def process_with_optimized_limits(meta, output_file, progress_output, max_files=None):
    """优化后的基因处理函数"""
    
    # 配置
    #progress_file = "D:/GeneAgent3/progress_%s.json" % (MODEL)
    print("Debugging - load progress file...")
    progress_file = progress_output
    batch_size = 3  # 更小的批次，更频繁保存
    
    # 初始化速率限制管理器
    rate_manager = RateLimitManager(tokens_per_minute=30000, safety_margin=0.7)  # 更保守的安全边际
    
    # 定义所有列
    original_columns = meta.columns.tolist()
    
    new_columns = [
    'Symbol (%s)' % MODEL,
    'Gene Name (%s)' % MODEL, 
    'Disease (%s)' % MODEL,
    'CHD Phenotype (%s)' % MODEL
    ]
    all_columns = original_columns + new_columns
    
    # 加载进度
    progress = load_progress(progress_file)
    processed_files = progress.get('processed_files', [])
    all_concat_rows = progress.get('all_concat_rows', [])
    
    print(f"📁 加载进度: 已处理 {len(processed_files)} 个文件")
    
    # 限制处理数量
    if max_files:
        meta = meta.head(max_files)
    
    total_files = len(meta)
    processed_count = len(processed_files)
    
    # 处理循环
    for index, row in meta.iterrows():
        basename = row["下载文件名"]
        gene_name = row["gene_name"]

        
        # 跳过已处理的文件
        if basename in processed_files:
            print(f"⏭️  跳过已处理文件: {basename}")
            continue
        
        if not basename or basename is None or (isinstance(basename, float) and math.isnan(basename)):
            print(basename)
            print(f"⚠️ 无有效文件名，自动跳过该条目")
            continue
        else:
            print("Basename is not none")
            print(basename)
            pdf_path = os.path.join(DIR_PATH, basename.strip())
        
            if not os.path.exists(pdf_path):
                print(f"❌ 文件不存在: {pdf_path}")
                continue    
        
        # 速率控制检查
        wait_time = rate_manager.should_wait()
        if wait_time > 0:
            print(f"⏰ 接近限制，主动等待 {wait_time:.1f} 秒... (当前速率: {rate_manager.get_current_rate()}/分钟)")
            time.sleep(wait_time)
        
        print(f"🔍 处理第 {index + 1}/{total_files} 个文件: {basename}")
        
        try:
            # 带重试的API调用
            output_text = synthesize_evidence_robust(pdf_path, gene_name)
            output_json = parse_response(output_text, MODEL)
            
            # 合并数据
            combined_dict = {**row.to_dict(), **output_json}
            all_concat_rows.append(combined_dict)
            processed_files.append(basename)
            
            # 记录成功请求
            rate_manager.record_request()
            
            processed_count += 1
            current_rate = rate_manager.get_current_rate()
            print(f"✅ 完成 {processed_count}/{total_files} - {basename} (当前速率: {current_rate}/分钟)")
            
            # 更频繁地保存进度
            save_progress(progress_file, processed_files, all_concat_rows)
            
            # 每batch_size个文件写入一次Excel
            if len(all_concat_rows) % batch_size == 0:
                print("💾 写入Excel...")
                current_df = pd.DataFrame(all_concat_rows)
                current_df = current_df.reindex(columns=all_columns, fill_value='')
                append_to_excel(output_file, current_df, all_columns)
            
            # 主动延迟 - 根据您的限制调整为8秒
            time.sleep(8)
            
        except Exception as e:
            print(f"❌ 处理失败 {basename}: {e}")
            
            # 保存当前进度
            save_progress(progress_file, processed_files, all_concat_rows)
            
            # 如果是严重的API错误，建议长时间等待
            if "rate_limit_exceeded" in str(e):
                wait_time = extract_wait_time_from_error(str(e))
                if wait_time and wait_time > 30:
                    print(f"🚨 遇到长时间限制 ({wait_time}秒)，建议暂停处理")
                    break
            
            continue
    
    # 处理完成后写入剩余数据
    if all_concat_rows:
        print("💾 写入最终数据...")
        current_df = pd.DataFrame(all_concat_rows)
        current_df = current_df.reindex(columns=all_columns, fill_value='')
        append_to_excel(output_file, current_df, all_columns)
        
        # 删除进度文件
        if os.path.exists(progress_file):
            #os.remove(progress_file)
            print("🧹 进度文件准备完成")
    
    final_count = len(all_concat_rows)
    print(f"🎉 处理完成! 总共处理了 {final_count} 个文件")
    return final_count

In [9]:
# rerun 20251112
import math
if __name__ == "__main__":
    MODEL = "gemini-2.5-pro"
    DIR_PATH = "D:/GeneAgent3/all_downloaded_pdf/"
    output_file = "D:/GeneAgent3/rerun_extracted_results_%s.xlsx" % MODEL
    progress_output = "D:/GeneAgent3/rerun_%s.json" % MODEL
    MAX_FILES = 500  # 最多处理文件数
    
    # 读取meta数据
    meta = pd.read_excel("D:/GeneAgent3/1029第二次运行补全/rerun/20251029_status_Gemini-2.5-pro_failed_entries.xlsx")
    #meta = meta.head(1)
    
    # 开始处理
    processed_count = process_with_optimized_limits(meta, output_file, progress_output, MAX_FILES)
    print(f"最终处理了 {processed_count} 个文件")

Debugging - load progress file...
🔒 速率限制配置:
   - 限额: 30000 tokens/分钟
   - 安全限额: 21000 tokens/分钟
   - 估算每个请求: 5000 tokens
   - 最大安全请求: 4 个/分钟
📁 加载进度: 已处理 22 个文件
⏭️  跳过已处理文件: AFF4_27125_10.1002_ajmg.a.61174.pdf
⏭️  跳过已处理文件: BCOR_54880_10.1038_ejhg.2009.52.pdf
⏭️  跳过已处理文件: CHD7_55636_10.1086_500273.pdf
⏭️  跳过已处理文件: CHST14_113189_10.1016_j.ajhg.2009.11.010.pdf
⏭️  跳过已处理文件: CHST14_113189_10.1002_ajmg.a.35613.pdf
⏭️  跳过已处理文件: DNAH11_8701_10.1002_uog.18915.pdf
⏭️  跳过已处理文件: EFTUD2_9343_10.1016_j.ajhg.2011.12.023.pdf
⏭️  跳过已处理文件: ELN_2006_10.1093_hmg_6.7.1021.pdf
⏭️  跳过已处理文件: FGFR2_2263_10.1038_ng0295-165.pdf
⏭️  跳过已处理文件: GATA4_2626_10.1002_ajmg.a.30684.pdf
⏭️  跳过已处理文件: KMT2A_4297_10.1016_j.ajhg.2012.06.008.pdf
⏭️  跳过已处理文件: PITX2_5308_10.1089_dna.2013.2185.pdf
⏭️  跳过已处理文件: RAB23_51715_10.1086_518047.pdf
⏭️  跳过已处理文件: RAD21_5885_10.1111_cge.12863.pdf
⏭️  跳过已处理文件: RAF1_5894_10.1038_ng2078.pdf
⏭️  跳过已处理文件: STRA6_64220_10.1086_518177.pdf
⏭️  跳过已处理文件: TFAP2B_7021_10.1089_gte.2008.0015.pdf
⏭️  跳过已处理文件

In [11]:
# 使用示例
import math
if __name__ == "__main__":
    MODEL = "gemini-2.5-pro"
    DIR_PATH = "D:/GeneAgent3/all_downloaded_pdf/"
    output_file = "D:/GeneAgent3/full_extracted_results_%s.xlsx" % MODEL
    progress_output = "D:/GeneAgent3/progress_%s.json" % MODEL
    MAX_FILES = 500  # 最多处理文件数
    
    # 读取meta数据
    meta = pd.read_excel("D:/GeneAgent3/处理结果/CHDgene_Metadata_最终下载状态_修正版.xlsx")
    #meta = meta.head(1)
    
    # 开始处理
    processed_count = process_with_optimized_limits(meta, output_file, progress_output, MAX_FILES)
    print(f"最终处理了 {processed_count} 个文件")

Debugging - load progress file...
🔒 速率限制配置:
   - 限额: 30000 tokens/分钟
   - 安全限额: 21000 tokens/分钟
   - 估算每个请求: 5000 tokens
   - 最大安全请求: 4 个/分钟
📁 加载进度: 已处理 0 个文件
Basename is not none
ABL1_25_10.1097_MD.0000000000014782.pdf
🔍 处理第 1/423 个文件: ABL1_25_10.1097_MD.0000000000014782.pdf
📖 正在处理: D:/GeneAgent3/all_downloaded_pdf/ABL1_25_10.1097_MD.0000000000014782.pdf
🔍 在第 4 页检测到参考文献标题: References

生成的文本： Symbol: ABL1
Name of gene: ABL proto-oncogene 1, non-receptor tyrosine kinase
Disease: Congenital heart defects, dysmorphic facial features, and skeletal anomalies syndrome
CHD Phenotype:
-Double outlet right ventricle;
-Pulmonary valve stenosis;
-Hypoplasia of the pulmonary trunk and branches;
{'Symbol gemini-2.5-pro': 'ABL1', 'Gene Name gemini-2.5-pro': 'ABL proto-oncogene 1, non-receptor tyrosine kinase', 'Disease gemini-2.5-pro': 'Congenital heart defects, dysmorphic facial features, and skeletal anomalies syndrome', 'CHD Phenotype gemini-2.5-pro': 'Double outlet right ventricle; Pulmonary val